In [ ]:
from netCDF4 import Dataset
import numpy as np
import pandas as pd
import os
from matplotlib import pyplot as plt
import fnmatch
import re

In [ ]:
def find_files(directory, pattern, maxdepth=None):
    flist = []
    for root, dirs, files in os.walk(directory):
        for basename in files:
            if fnmatch.fnmatch(basename, pattern):
                filename = os.path.join(root, basename)
                filename = filename.replace('\\\\', os.sep)
                if maxdepth is None:
                    flist.append(filename)
                else:
                    if filename.count(os.sep)-directory.count(os.sep) <= maxdepth:
                        flist.append(filename)
    return flist

In [ ]:
R  = 6400

def distance(lat1, lon1, lat2, lon2):
    lat_av = np.deg2rad(lat1 - lat2)/2
    lon_av = np.deg2rad(lon1 - lon2)/2
    dd = np.sin(lat_av)**2 + np.cos(np.deg2rad(lat2)) * np.cos(np.deg2rad(lat1)) * np.sin(lon_av)**2
    dd = 2*R*np.arcsin(np.sqrt(dd))
    return dd

In [ ]:
year = 2015
deltaT = 6

In [ ]:
files = find_files(f'/mnt/hippocamp/DATA/satellite/SMAP_V6.0/L2C/{year}/', '*.nc')
files.sort()
len(files)

In [ ]:
insitu = pd.read_csv(f'/mnt/hippocamp/asavin/data/sss_insitu/Data_insitu_{year}.csv')

In [ ]:
insitu

In [ ]:
insitu['Time'] = pd.to_datetime(insitu['Time'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

In [ ]:
file_info = []

for fname in files:
    m = re.search(r'(\d{8}T\d{6})', fname)
    if m:
        file_time = pd.to_datetime(m.group(1), format='%Y%m%dT%H%M%S')
        file_info.append((fname, file_time))

In [ ]:
delta = pd.Timedelta(hours=deltaT)

def find_files_for_time(t):
    if pd.isna(t):
        return ''
    
    matched = [
        fname for fname, ftime in file_info
        if abs(ftime - t) <= delta
    ]
    
    return '; '.join(matched)

In [ ]:
insitu['matched_files'] = insitu['Time'].apply(find_files_for_time)

In [ ]:
insitu

In [ ]:
print(insitu.loc[0, 'matched_files'])